In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [ ]:
current_dir = Path(os.path.dirname(os.path.abspath('__file__'))).parent
path = os.path.join(current_dir, "data/raw/ccnews/articles_filtered.jsonl")

# Read JSONL
df = pd.read_json(path, lines=True)

print("total unique urls:", df['url'].nunique())

lang = "ru"


# Keep lang only (adjust if you store different lang format)
df["lang"] = df["lang"].astype(str).str.lower().str.strip()
df_en = df[df["lang"]==lang].copy()

# Clean key fields
df_en["domain"] = df_en["domain"].astype(str).str.lower().str.strip()
df_en["url"] = df_en["url"].astype(str).str.strip()
df_en = df_en[(df_en["domain"] != lang) & (df_en["url"] != "")]

# Group by domain and count unique URLs
domain_stats = (
    df_en.groupby("domain", as_index=False)
    .agg(unique_urls=("url", "nunique"))
    .sort_values("unique_urls", ascending=False)
    .reset_index(drop=True)
)

# Share and cumulative share
total_unique_urls = domain_stats["unique_urls"].sum()
domain_stats["share_pct"] = domain_stats["unique_urls"] / total_unique_urls * 100
domain_stats["cum_share_pct"] = domain_stats["share_pct"].cumsum()

# Domains covering ~80%
top80 = domain_stats[domain_stats["cum_share_pct"] <= 80].copy()
if len(top80) < len(domain_stats):
    top80 = domain_stats.iloc[: len(top80) + 1].copy()

print(f"Total {lang} rows:", len(df_en))
print(f"Total {lang} unique URLs:", total_unique_urls)
print(f"Unique {lang} domains:", domain_stats["domain"].nunique())
print(f"Domains for ~80% coverage:", len(top80))
print(f"Coverage reached:", round(top80["share_pct"].sum(), 2), "%")

display(domain_stats.head(30))
display(top80)

In [ ]:
# assumes df_en already exists and contains at least: date, url
tmp = df_en.copy()

# parse dates safely
tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
tmp = tmp[tmp["date"] > "2020-01-01"]
tmp = tmp.dropna(subset=["date", "url"])

# month bucket
tmp["month"] = tmp["date"].dt.to_period("M").dt.to_timestamp()

# monthly unique URLs
monthly_unique = (
    tmp.groupby("month")["url"]
    .nunique()
    .reset_index(name="unique_articles")
    .sort_values("month")
)

display(monthly_unique)

# plot
plt.figure(figsize=(10, 4))
plt.plot(monthly_unique["month"], monthly_unique["unique_articles"], marker="o")
plt.title("Monthly unique EN articles")
plt.xlabel("Month")
plt.ylabel("Unique articles")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# list of en domains
['www.aljazeera.com',
'www.channelnewsasia.com',
'www.thenationalnews.com',
'www.swissinfo.ch',
'www.straitstimes.com',
'vietnamnews.vn',
'abcnews.go.com',
'www.cbsnews.com',
'www.foxnews.com',
'www.latimes.com',
'globalnews.ca',
'www.ctvnews.ca',
'www.telegraph.co.uk',
'www.independent.co.uk',
'www.the-independent.com',
'www.irishtimes.com',
'www.scotsman.com',
'www.yorkshirepost.co.uk',
'timesofindia.indiatimes.com',
'economictimes.indiatimes.com',
'indianexpress.com',
'www.ndtv.com',
'www.business-standard.com',
'www.livemint.com',
'www.businesstoday.in',
'gulfnews.com',
'www.khaleejtimes.com',
'allafrica.com',
'www.timeslive.co.za',
'www.businesslive.co.za',
'thewest.com.au',
'www.nzherald.co.nz',
'www.nasdaq.com',
'seekingalpha.com',
'qz.com',
'www.barchart.com',
'www.pymnts.com']

In [ ]:
# ru domains
[
'tass.ru',
'ria.ru',
'www.interfax.ru',
'www.rbc.ru',
'www.vedomosti.ru',
'www.kommersant.ru',
'lenta.ru',
'www.nur.kz',
'www.zakon.kz'
]

## check progress

In [ ]:
range(2025)

In [ ]:
month

In [ ]:
from collections import Counter
import pandas as pd
from src.collectors.ccnews_extractor import (
    load_processed,
    load_month_paths,
    Path,
    os
)


current_dir = Path(os.path.dirname(os.path.abspath('__file__'))).parent
PROCESSED_LOG = os.path.join(current_dir, "data/raw/ccnews/processed_warcs_filtered.json")
BASE_URL = "https://data.commoncrawl.org/"
MONTHS = [f'{year}/{month:02d}' for year in [2025] for month in range(1, 13)]
MONTHS.extend(['2026/01','2026/02','2026/03'])

processed = load_processed(PROCESSED_LOG)
processed_set = set(processed)

rows = []
for month in MONTHS:
    month_paths = load_month_paths(BASE_URL, month)
    total = len(month_paths)
    done = sum(1 for p in month_paths if p in processed_set)
    left = total - done
    pct = (done / total * 100) if total else 0.0
    rows.append(
        {
            "month": month,
            "total_warcs": total,
            "processed_warcs": done,
            "remaining_warcs": left,
            "processed_pct": round(pct, 2),
        }
    )

progress_df = pd.DataFrame(rows).sort_values("month").reset_index(drop=True)

year_total = int(progress_df["total_warcs"].sum())
year_done = int(progress_df["processed_warcs"].sum())
year_left = int(progress_df["remaining_warcs"].sum())
year_pct = round((year_done / year_total * 100) if year_total else 0.0, 2)

print(f"processed_log_entries={len(processed_set)}")
print(f"year_total_warcs={year_total}")
print(f"year_processed_warcs={year_done}")
print(f"year_remaining_warcs={year_left}")
print(f"year_processed_pct={year_pct}%")

progress_df

In [ ]:
import pandas as pd
from pathlib import Path
import os
from src.collectors.ccnews_extractor import load_processed, load_month_paths

current_dir = Path(os.path.dirname(os.path.abspath('__file__'))).parent
DATA_DIR = Path(current_dir) / "data/raw/ccnews"
PROCESSED_LOG = DATA_DIR / "processed_warcs_filtered.json"
BASE_URL = "https://data.commoncrawl.org/"
MONTHS = [f"{year}/{month:02d}" for year in [2025] for month in range(1, 13)]
MONTHS.extend(["2026/01", "2026/02", "2026/03"])

processed = load_processed(str(PROCESSED_LOG))
processed_set = set(processed)

progress_rows = []
file_rows = []
dfs = []

for month in MONTHS:
    month_paths = load_month_paths(BASE_URL, month)
    total = len(month_paths)
    done = sum(1 for p in month_paths if p in processed_set)
    left = total - done
    pct = (done / total * 100) if total else 0.0

    year, mon = month.split("/")
    month_dir = DATA_DIR / year / mon
    files = []
    if month_dir.exists():
        files = sorted(list(month_dir.glob("*.jsonl")) + list(month_dir.glob("*.parquet")))

    month_rows = 0
    for fp in files:
        part = pd.read_parquet(fp) if fp.suffix == ".parquet" else pd.read_json(fp, lines=True)
        part["source_file"] = str(fp)
        month_rows += len(part)
        dfs.append(part)

    progress_rows.append(
        {
            "month": month,
            "total_warcs": total,
            "processed_warcs": done,
            "remaining_warcs": left,
            "processed_pct": round(pct, 2),
        }
    )
    file_rows.append({"month": month, "files_found": len(files), "rows_found": month_rows})

legacy_path = DATA_DIR / "articles_filtered.jsonl"
if legacy_path.exists() and legacy_path.stat().st_size > 0:
    legacy_df = pd.read_json(legacy_path, lines=True)
    legacy_df["source_file"] = str(legacy_path)
    dfs.append(legacy_df)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
if "url" in df.columns:
    df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

progress_df = pd.DataFrame(progress_rows).sort_values("month").reset_index(drop=True)
files_df = pd.DataFrame(file_rows).sort_values("month").reset_index(drop=True)

year_total = int(progress_df["total_warcs"].sum())
year_done = int(progress_df["processed_warcs"].sum())
year_left = int(progress_df["remaining_warcs"].sum())
year_pct = round((year_done / year_total * 100) if year_total else 0.0, 2)

print(f"processed_log_entries={len(processed_set)}")
print(f"year_total_warcs={year_total}")
print(f"year_processed_warcs={year_done}")
print(f"year_remaining_warcs={year_left}")
print(f"year_processed_pct={year_pct}%")
print(f"files_found_total={int(files_df['files_found'].sum())}")
print(f"rows_found_total={int(files_df['rows_found'].sum())}")
print(f"df_rows_unique_urls={len(df)}")

progress_df.merge(files_df, on="month", how="left")

In [ ]:
df.groupby('domain').agg(
    unique_urls=('url', 'nunique')
).sort_values('unique_urls', ascending=False)
